# Univariate covariate screening for the per-pair Weibull headway model

Each of the 6 pairs already has a preliminary Weibull fit. Here we add **one covariate
at a time** to that Weibull (via the AFT scale, \(\lambda_i=\exp(\beta_0+\beta_1 z_i)\))
and test whether it improves the distribution, using a likelihood-ratio test and AIC
against the covariate-free (intercept-only) Weibull.

**Candidates screened:** subject speed, speed-difference, flow, off-centeredness.
Continuous covariates are mean-centred within pair; off-centeredness enters as True/False.

**Reporting units:** speed & speed-difference per +1 km/h; flow per +1000 pcu/hr;
off-centeredness as True-vs-False. A pair/covariate with too little variation
(or a rare binary group) is flagged and skipped.

This is a *screening* step — covariates are tested singly. Because speed and
speed-difference are collinear (speed-difference = subject − leader speed), only one
should enter any combined model later. Output goes to the `Tables` folder.

In [1]:
# --- Cell 1: Imports and paths ---
import os
import numpy as np
import pandas as pd
from scipy import stats, optimize

BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data2.xlsx")
TABLES    = os.path.join(BASE, "Tables")
os.makedirs(TABLES, exist_ok=True)

In [2]:
# --- Cell 2: Load and drop the 2W pairs ---
df = pd.read_excel(DATA_PATH)
df = df[df["V_Leading_Class"] != "2W"].copy()
df = df.rename(columns={"Time_Headway": "hw"})
print("N =", len(df), "| pairs:", sorted(df["Pair"].unique()))

# covariate registry: name -> (column, type, unit, unit_label)
CANDIDATES = {
    "speed":      ("Target_Speed_km/hr", "cont", 1,    "per +1 km/h"),
    "speed_diff": ("Speed_Difference",   "cont", 1,    "per +1 km/h"),
    "flow":       ("Flow_pcu/hr",        "cont", 1000, "per +1000 pcu/hr"),
    "off_cen":    ("Off_centeredness",   "bin",  1,    "True vs False"),
}
MIN_BIN_GROUP = 5   # minimum count in each binary group to attempt a fit

N = 802 | pairs: ['BTW_following_4W', 'BTW_following_MT_3W', 'BTW_following_NMT_3W', 'PR_following_4W', 'PR_following_MT_3W', 'PR_following_NMT_3W']


In [3]:
# --- Cell 3: Weibull AFT likelihood + fitters ---
def _nll_cov(p, t, z):
    lp = stats.weibull_min.logpdf(t, np.exp(p[0]), loc=0, scale=np.exp(p[1] + p[2] * z))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12

def _nll_null(p, t):
    lp = stats.weibull_min.logpdf(t, np.exp(p[0]), loc=0, scale=np.exp(p[1]))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12

def fit_null(t):
    c0, _, s0 = stats.weibull_min.fit(t, floc=0)
    r = optimize.minimize(_nll_null, [np.log(c0), np.log(s0)], args=(t,),
                          method="Nelder-Mead", options={"maxiter": 8000})
    return r, (np.log(c0), np.log(s0))

def fit_cov(t, z, init):
    r = optimize.minimize(_nll_cov, [init[0], init[1], 0.0], args=(t, z),
                          method="Nelder-Mead",
                          options={"maxiter": 8000, "xatol": 1e-7, "fatol": 1e-7})
    return r

In [4]:
# --- Cell 4: Screen every candidate against each pair's null Weibull ---
rows = []
for pair, g in df.groupby("Pair"):
    t = g["hw"].values
    n = len(t)
    r0, init = fit_null(t)
    ll0 = -r0.fun
    shape_c = np.exp(r0.x[0])

    for name, (col, typ, unit, ulab) in CANDIDATES.items():
        raw = g[col].values.astype(float)

        # prepare covariate and check variation
        if typ == "cont":
            z = raw - raw.mean()
            enough = raw.std() > 1e-9
        else:  # binary
            z = raw
            enough = (len(np.unique(raw)) == 2 and
                      min((raw == 0).sum(), (raw == 1).sum()) >= MIN_BIN_GROUP)

        if not enough:
            rows.append({"Pair": pair, "N": n, "shape_c": round(shape_c, 3),
                         "Covariate": name, "Effect": "-- low variation --",
                         "coef_b1": np.nan, "LR_chi2": np.nan, "LR_p": np.nan,
                         "dAIC": np.nan, "improves": "n/a"})
            continue

        r1 = fit_cov(t, z, init)
        ll1 = -r1.fun
        b1 = r1.x[2]
        LR = 2 * (ll1 - ll0)
        p_lr = stats.chi2.sf(LR, 1)
        dAIC = (2 * 2 - 2 * ll0) - (2 * 3 - 2 * ll1)   # null - covariate; >0 favours covariate
        pct = (np.exp(b1 * unit) - 1) * 100
        eff = f"{pct:+.2f}%  {ulab}"

        rows.append({"Pair": pair, "N": n, "shape_c": round(shape_c, 3),
                     "Covariate": name, "Effect": eff, "coef_b1": round(b1, 5),
                     "LR_chi2": round(LR, 2),
                     "LR_p": "<0.001" if p_lr < 0.001 else round(p_lr, 4),
                     "dAIC": round(dAIC, 2),
                     "improves": "Yes" if p_lr < 0.05 else "No"})

screening = pd.DataFrame(rows)
screening

,Pair,N,shape_c,Covariate,Effect,coef_b1,LR_chi2,LR_p,dAIC,improves
0,BTW_following_4W,250,2.906,speed,-1.99% per +1 km/h,-0.02010,28.78,<0.001,26.78,Yes
1,BTW_following_4W,250,2.906,speed_diff,-2.50% per +1 km/h,-0.02532,31.76,<0.001,29.76,Yes
2,BTW_following_4W,250,2.906,flow,+2.71% per +1000 pcu/hr,0.00003,6.25,0.0124,4.25,Yes
3,BTW_following_4W,250,2.906,off_cen,-9.21% True vs False,-0.09659,3.83,0.0504,1.83,No
4,BTW_following_MT_3W,186,2.303,speed,-3.15% per +1 km/h,-0.03197,36.26,<0.001,34.26,Yes
5,BTW_following_MT_3W,186,2.303,speed_diff,-3.32% per +1 km/h,-0.03379,28.81,<0.001,26.81,Yes
6,BTW_following_MT_3W,186,2.303,flow,+5.97% per +1000 pcu/hr,0.00006,16.13,<0.001,14.13,Yes
7,BTW_following_MT_3W,186,2.303,off_cen,-4.92% True vs False,-0.05047,0.59,0.4438,-1.41,No
8,BTW_following_NMT_3W,160,2.286,speed,-2.12% per +1 km/h,-0.02139,8.37,0.0038,6.37,Yes
9,BTW_following_NMT_3W,160,2.286,speed_diff,-0.95% per +1 km/h,-0.00952,1.71,0.1913,-0.29,No


In [5]:
# --- Cell 5: Compact matrices for eyeballing ---
order_cov = list(CANDIDATES.keys())
order_pair = (screening.drop_duplicates("Pair").sort_values("N", ascending=False)["Pair"].tolist())

# numeric dAIC for the pivot (NaN where skipped)
num = screening.copy()
num["dAIC_num"] = pd.to_numeric(num["dAIC"], errors="coerce")

dAIC_matrix = (num.pivot(index="Pair", columns="Covariate", values="dAIC_num")
                  .reindex(index=order_pair, columns=order_cov).round(2))

def p_to_num(v):
    return 0.0005 if v == "<0.001" else (np.nan if pd.isna(v) else float(v))
num["p_num"] = num["LR_p"].map(p_to_num)
LRp_matrix = (num.pivot(index="Pair", columns="Covariate", values="p_num")
                 .reindex(index=order_pair, columns=order_cov))

print("dAIC (null - covariate); positive => covariate improves the fit")
print(dAIC_matrix.to_string())
dAIC_matrix

dAIC (null - covariate); positive => covariate improves the fit
Covariate             speed  speed_diff   flow  off_cen
Pair                                                   
BTW_following_4W      26.78       29.76   4.25     1.83
BTW_following_MT_3W   34.26       26.81  14.13    -1.41
BTW_following_NMT_3W   6.37       -0.29   3.77    -2.00
PR_following_MT_3W    16.08        1.13   2.91    -1.93
PR_following_NMT_3W   -1.86       -1.22  -1.03    -1.38
PR_following_4W       -1.99       -1.99  -1.98     0.37


Covariate,speed,speed_diff,flow,off_cen
Pair,,,,
BTW_following_4W,26.78,29.76,4.25,1.83
BTW_following_MT_3W,34.26,26.81,14.13,-1.41
BTW_following_NMT_3W,6.37,-0.29,3.77,-2.00
PR_following_MT_3W,16.08,1.13,2.91,-1.93
PR_following_NMT_3W,-1.86,-1.22,-1.03,-1.38
PR_following_4W,-1.99,-1.99,-1.98,0.37


In [6]:
# --- Cell 6: Best single covariate per pair ---
best = []
for pair in order_pair:
    sub = num[(num["Pair"] == pair) & num["dAIC_num"].notna()]
    if sub.empty:
        best.append({"Pair": pair, "Best_covariate": "none", "dAIC": np.nan,
                     "LR_p": np.nan, "significant": "No"}); continue
    top = sub.loc[sub["dAIC_num"].idxmax()]
    best.append({"Pair": pair, "N": int(top["N"]),
                 "Best_covariate": top["Covariate"], "dAIC": round(top["dAIC_num"], 2),
                 "LR_p": top["LR_p"],
                 "significant": "Yes" if p_to_num(top["LR_p"]) < 0.05 else "No"})
best_covariate = pd.DataFrame(best)
best_covariate

,Pair,N,Best_covariate,dAIC,LR_p,significant
0,BTW_following_4W,250,speed_diff,29.76,<0.001,Yes
1,BTW_following_MT_3W,186,speed,34.26,<0.001,Yes
2,BTW_following_NMT_3W,160,speed,6.37,0.0038,Yes
3,PR_following_MT_3W,104,speed,16.08,<0.001,Yes
4,PR_following_NMT_3W,59,flow,-1.03,0.3239,No
5,PR_following_4W,43,off_cen,0.37,0.124,No


In [7]:
# --- Cell 7: Save all screening outputs to the Tables folder ---
out_path = os.path.join(TABLES, "headway_covariate_screening.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    screening.to_excel(xl,      sheet_name="Screening_long", index=False)
    dAIC_matrix.to_excel(xl,    sheet_name="dAIC_matrix")
    LRp_matrix.round(4).to_excel(xl, sheet_name="LR_p_matrix")
    best_covariate.to_excel(xl, sheet_name="Best_per_pair", index=False)
print("Saved:", out_path)

Saved: D:\Headway\Tables\headway_covariate_screening.xlsx
